In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.distributions.uniform import Uniform #initialize tensors with uniform distribution
from torch.utils.data import TensorDataset, DataLoader
import lightning as L
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Creating the Dataset

In [2]:
inputs = torch.tensor([[1., 0., 0., 0.],  # one-hot-encoding for Troll 2
                       [0., 1., 0., 0.],  # one-hot-encoding for is
                       [0., 0., 1., 0.],  # one-hot-encoding for great
                       [0., 0., 0., 1.]]) # one-hot-encoding for Gymkata

labels = torch.tensor([[0., 1., 0., 0.],
                       [0., 0., 1., 0.],
                       [0., 0., 0., 1.],
                       [0., 1., 0., 0.]])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Build and Train a Word Embedding using nn.Linear()

In [3]:
class WordEmbedding(L.LightningModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        L.seed_everything(seed=42)
        self.input_hidden = nn.Linear(in_features=4, out_features=2, bias=False)
        self.hidden_output = nn.Linear(in_features=2, out_features=4, bias=False)
        self.loss = nn.CrossEntropyLoss()

    def forward(self, input):
        hidden = self.input_hidden(input)
        output = self.hidden_output(hidden)
        return output
    

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch):
        input_i, label_i = batch
        output_i = self.forward(input_i)
        loss = self.loss(output_i, label_i)
        return loss

In [4]:
model = WordEmbedding()
trainer = L.Trainer(max_epochs=100)
trainer.fit(model, train_dataloaders=dataloader)

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name          | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | input_hidden  | Linear           | 8      | train | 0    
1 | hidden_output | Linear           | 8      | train | 0    
2 | loss          | CrossEntropyLoss | 0      | train | 0    
-------------------------------------------------------------------
16        Trainable params
0         Non-trainable params
16        Total params
0.000

Epoch 99: 100%|██████████| 4/4 [00:00<00:00, 508.57it/s, v_num=2]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 4/4 [00:00<00:00, 369.03it/s, v_num=2]


In [5]:
# Model predictions
softmax = nn.Softmax(dim=1) # dim=0 applies softmax to rows, dim=1 applies softmax to columns



print(torch.round(softmax(model(torch.tensor([[1., 0., 0., 0.]]))), decimals=2)) # print the predictions for "Troll2"
print(torch.round(softmax(model(torch.tensor([[0., 1., 0., 0.]]))), decimals=2)) # print the predictions for "is"
print(torch.round(softmax(model(torch.tensor([[0., 0., 1., 0.]]))), decimals=2)) # print the predictions for "great"
print(torch.round(softmax(model(torch.tensor([[0., 0., 0., 1.]]))), decimals=2)) # print the predictions for "Gymkata"

tensor([[0., 1., 0., 0.]], grad_fn=<RoundBackward1>)
tensor([[0., 0., 1., 0.]], grad_fn=<RoundBackward1>)
tensor([[0., 0., 0., 1.]], grad_fn=<RoundBackward1>)
tensor([[0., 1., 0., 0.]], grad_fn=<RoundBackward1>)
